# Assignment 12: Predicting Hotel Booking Cancellations  
## Models: Naïve Bayes, Support Vector Machine (SVM), and Neural Network

**Objectives:**
- Understand how to use classification models (Naïve Bayes, SVM, Neural Networks) to predict hotel cancellations.
- Compare models in terms of accuracy, complexity, and business relevance.
- Interpret and communicate model results from a business perspective.

<a href="https://colab.research.google.com/github/Stan-Pugsley/is_4487_base/blob/main/Assignments/assignment_12_bayes_svm_neural.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

## Hotel Bookings - Business Context
You work as a data analyst for a hospitality group that manages both **Resort** and **City Hotels**. One major challenge in operations is the unpredictability of **booking cancellations**, which affects staffing, inventory, and revenue planning.

You’ve been asked to use historical booking data to predict whether a future booking will be canceled. Your insights will help management plan more effectively.

Your tasks are to:
1. Build and evaluate three models: Naïve Bayes, SVM, and Neural Network.
2. Compare performance.
3. Recommend which model is best suited for the business needs.

### Key Use Cases
- Understand customer booking behavior
- Explore factors related to cancellations
- Segment guests based on booking characteristics
- Compare city vs. resort hotel performance




## Data Dictionary

This dataset contains booking information for two types of hotels: a **city hotel** and a **resort hotel**. Each record corresponds to a single booking and includes various details about the reservation, customer demographics, booking source, and whether the booking was canceled.

**Source**: [GitHub - TidyTuesday: Hotel Bookings](https://github.com/rfordatascience/tidytuesday/blob/master/data/2020/2020-02-11/readme.md)

| Variable | Type | Description |
|----------|------|-------------|
| `hotel` | character | Hotel type: City or Resort |
| `is_canceled` | integer | 1 = Canceled, 0 = Not Canceled |
| `lead_time` | integer | Days between booking and arrival |
| `arrival_date_year` | integer | Year of arrival |
| `arrival_date_month` | character | Month of arrival |
| `stays_in_weekend_nights` | integer | Nights stayed on weekends |
| `stays_in_week_nights` | integer | Nights stayed on weekdays |
| `adults` | integer | Number of adults |
| `children` | integer | Number of children |
| `babies` | integer | Number of babies |
| `meal` | character | Type of meal booked |
| `country` | character | Country code of origin |
| `market_segment` | character | Booking source (e.g., Direct, Online TA) |
| `distribution_channel` | character | Booking channel used |
| `is_repeated_guest` | integer | 1 = Repeated guest, 0 = New guest |
| `previous_cancellations` | integer | Past booking cancellations |
| `previous_bookings_not_canceled` | integer | Past bookings not canceled |
| `reserved_room_type` | character | Initially reserved room type |
| `assigned_room_type` | character | Room type assigned at check-in |
| `booking_changes` | integer | Number of booking modifications |
| `deposit_type` | character | Deposit type (No Deposit, Non-Refund, etc.) |
| `agent` | character | Agent ID who made the booking |
| `company` | character | Company ID (if booking through company) |
| `days_in_waiting_list` | integer | Days on the waiting list |
| `customer_type` | character | Booking type: Contract, Transient, etc. |
| `adr` | float | Average Daily Rate (price per night) |
| `required_car_parking_spaces` | integer | Requested parking spots |
| `total_of_special_requests` | integer | Number of special requests made |
| `reservation_status` | character | Final status (Canceled, No-Show, Check-Out) |
| `reservation_status_date` | date | Date of the last status update |

This dataset is ideal for classification, segmentation, and trend analysis exercises.


## 1. Load and Prepare the Hotel Booking Dataset

**Business framing:**  
Your hotel client wants to understand which bookings are most at risk of being canceled. But before modeling, your job is to prepare the data to ensure clean and reliable input.

### Do the following:
- Import data from the hotels dataset into a dataframe (in GitHub go to the DataSets folder and look for `hotels.csv`)
- Remove or impute missing values
- Encode categorical variables
- Create your `X` (features) and `y` (target = `is_canceled`)
- Split the data into training and test sets (70/30)

**Important:** Perform this split **before** any preprocessing or feature transformations.

### Reflection:
1. How many total rows and columns are in the dataset?
2. What types of features (categorical, numerical) are included?
3. What steps did you take to clean or prepare the data?


In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# 1. Load the dataset from the course GitHub repository
df = pd.read_csv("hotels.csv")

print(f"Dataset shape: {df.shape[0]} rows, {df.shape[1]} columns")
print("\nData types:\n", df.dtypes.value_counts())

# 2. Drop leakage columns and high-cardinality ID columns that act like noise
cols_to_drop = ['reservation_status', 'reservation_status_date', 'agent', 'company']
df_clean = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

# 3. Define X (features) and y (target)
X = df_clean.drop(columns=['is_canceled'])
y = df_clean['is_canceled']

# 4. Split data BEFORE any preprocessing (70% train, 30% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

# 5. Identify numerical and categorical columns
num_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

# 6. Build preprocessing pipelines (fitted ONLY on X_train)
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

# Fit on training data, transform both train and test
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print(f"\nProcessed training shape: {X_train_processed.shape}")
print(f"Processed test shape: {X_test_processed.shape}")


Dataset shape: 9404 rows, 32 columns

Data types:
 int64      17
object     12
float64     3
Name: count, dtype: int64

Processed training shape: (6582, 137)
Processed test shape: (2822, 137)


### ✍️ Your Response: 🔧
1. The dataset contains 9,404 rows and 32 columns. One hot encoding expands columns to 137.

2. There are 20 numerical features and 12 categorical features. Numerical comes in integer and decimal columns (int and float).

3. I removed columns causing data leakage such as reservation status and reservation date. I also performed a 70/30 train-test split on the data. I also imputed some missing values, I filled numerical values with the median and categorical values with the mode.

## 2. Build a Naïve Bayes Model

**Business framing:**  
Naïve Bayes is a quick, baseline model often used for early testing or simple classification problems.

### Do the following:
- Make sure to split your data first (see the previous step), then fit any text/vector preprocessing on training data only.
- Train a Naïve Bayes classifier on your training data
- Use it to predict on your test data
- Print a classification report and confusion matrix

**Note:** If you use a vectorizer (e.g., `CountVectorizer`), fit it on the training data only, then transform both training and test data.


In [4]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# 1. Initialize and train the Naïve Bayes classifier on training data
nb_model = GaussianNB()
nb_model.fit(X_train_processed, y_train)

# 2. Predict on the test data
y_pred_nb = nb_model.predict(X_test_processed)
nb_acc = accuracy_score(y_test, y_pred_nb)

# 3. Print accuracy, confusion matrix, and classification report
print(f"Naïve Bayes Accuracy: {nb_acc:.4f}\n")
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_nb))
print("\nClassification Report:\n", classification_report(y_test, y_pred_nb))


Naïve Bayes Accuracy: 0.4968

Confusion Matrix:
 [[ 702 1419]
 [   1  700]]

Classification Report:
               precision    recall  f1-score   support

           0       1.00      0.33      0.50      2121
           1       0.33      1.00      0.50       701

    accuracy                           0.50      2822
   macro avg       0.66      0.66      0.50      2822
weighted avg       0.83      0.50      0.50      2822



### Reflection:
1. How well does the model perform?  And what metric is best used to judge the performance?
2. Where might this model be useful for the hotel (e.g. real-time alerts, operational decisions)?

### ✍️ Your Response: 🔧
1. The naive bayes model performs poorly as a standalone predictor, achieving 49.7% accuracy, which is actually worse than if you were to just predict not canceled for every booking. The confusion matrix shows that it correctly identified 700 out of 701 actual cancellations, but produced 1,419 false positives. The datasate is imbalanced so raw accuracy is misleeading. F1 score and preceision/recall score for canceled classes are better metrics to judge performance.

2. The model could be useful as a fast, low-cost initial screening filter or real-time alert system when new bookings arrive. Due to the high false positive rates, it should not be used to automatically overbook rooms. Knowing the capabilities of the model and knowing when to and not to use it are crucial.

## 3. Build a Support Vector Machine (SVM) Model

**Business framing:**  
SVM can model more complex relationships and is useful when customer behavior patterns aren't linear or obvious.

### Do the following:
- Scale the data using `StandardScaler` to bring large numbers into a smaller range (Note: use a scaled training set, but fit the scaler only on X_train).
- Train an SVM classifier (use `linear` kernel)
- Make predictions and evaluate with classification metrics

**NOTE:** With about 10K rows, this model may run very **slow**.  Be prepared to wait up to 10 minutes.   

In [6]:
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# 1. Initialize and train the SVM classifier with a linear kernel
# (X_train_processed was already scaled using StandardScaler fitted only on X_train in Section 1)
svm_model = SVC(kernel='linear', random_state=42)
svm_model.fit(X_train_processed, y_train)

# 2. Make predictions on the scaled test set
y_pred_svm = svm_model.predict(X_test_processed)
svm_acc = accuracy_score(y_test, y_pred_svm)

# 3. Evaluate with classification metrics
print(f"SVM Accuracy: {svm_acc:.4f}\n")
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_svm))
print("\nClassification Report:\n", classification_report(y_test, y_pred_svm))

SVM Accuracy: 0.9018

Confusion Matrix:
 [[1994  127]
 [ 150  551]]

Classification Report:
               precision    recall  f1-score   support

           0       0.93      0.94      0.94      2121
           1       0.81      0.79      0.80       701

    accuracy                           0.90      2822
   macro avg       0.87      0.86      0.87      2822
weighted avg       0.90      0.90      0.90      2822



### Reflection:
1. How well does the model perform?  And what metric is best used to judge the performance?
2. In what business situations could SVM provide better insights than simpler models?

### ✍️ Your Response: 🔧
1. The linear SVM performs very well, achieving 90.2% accuracy and which greatly outperforms the Naive Bayes baseline. Looking at the confusion matrix, it accurately identified 551 out of 701 actual cancellations while reducing false positives to just 127. Because the dataset is imbalanced, false positives and false negatives carry operational costs, F1 score and precision/recall score are the best metrics to judge performance.

2. SVM can be more reliable for revenue management, purposeful overbooking, and dynamic pricing. These are all strategies where management needs high confidence that a booking will actually cancel.

## 4. Build a Neural Network Model

**Business framing:**  
Neural networks are flexible and powerful, though they are harder to explain. They may work well when subtle patterns exist in the data.

### Do the following:
- Build a MLPClassifier model using the neural_network package from sklearn
- Choose a simple architecture (e.g., 2 hidden layers)
- Use a true validation split from the training data, not the test set, for validation_data
- Evaluate accuracy and performance

**NOTE:** With about 10K rows, this model may run very **slow**.  Be prepared to wait up to 10 minutes.  

In [7]:
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# 1. Create a true validation split strictly from the training data (never touching X_test)
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_processed, y_train, test_size=0.15, random_state=42, stratify=y_train
)

# 2. Build an MLPClassifier with 2 hidden layers (64 and 32 neurons)
mlp_model = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation='relu',
    solver='adam',
    max_iter=100,
    random_state=42
)

# Train on the sub-training set and check validation accuracy
mlp_model.fit(X_tr, y_tr)
val_acc = accuracy_score(y_val, mlp_model.predict(X_val))
print(f"Validation Accuracy (from training split): {val_acc:.4f}")

# 3. Evaluate final accuracy and performance on the unseen test set
y_pred_mlp = mlp_model.predict(X_test_processed)
mlp_acc = accuracy_score(y_test, y_pred_mlp)

print(f"Neural Network Test Accuracy: {mlp_acc:.4f}\n")
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_mlp))
print("\nClassification Report:\n", classification_report(y_test, y_pred_mlp))


Validation Accuracy (from training split): 0.9342
Neural Network Test Accuracy: 0.9174

Confusion Matrix:
 [[1995  126]
 [ 107  594]]

Classification Report:
               precision    recall  f1-score   support

           0       0.95      0.94      0.94      2121
           1       0.82      0.85      0.84       701

    accuracy                           0.92      2822
   macro avg       0.89      0.89      0.89      2822
weighted avg       0.92      0.92      0.92      2822



/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the optimization hasn't converged yet.
  warnings.warn(


### Reflection:
1. How does this model compare to the others?
2. Would the business be comfortable using a “black box” model like this? Why or why not?

### ✍️ Your Response: 🔧
1. The Neural Network model achieved the bets overall performance of all three models, reaching a 91.7% accuracy. It also improved cancellation recall from 0.79 to 0.85, while keeping false positives steady. This shows hidden layers successfully captured subtle, nonlinear combinations of booking attributes that linear models missed.

2. Hotel management would likely be comfortable using it for automated, behind the scenes tasks, where maximizing predictive accuracy directly protects revenue. There may be more hesitation to use it for customer-facing decisions because these types of models cannot explain why specific data points, in this case bookings, were flagged.

## 5. Compare All Three Models

### Do the following:
- Print and compare the accuracy of Naïve Bayes, SVM, and Neural Network models
- Summarize which model performed best

In [8]:
# Create a comparison table using your accuracy variables from Sections 2, 3, and 4
comparison_df = pd.DataFrame({
    'Model': ['Naïve Bayes', 'Support Vector Machine (SVM)', 'Neural Network (MLP)'],
    'Test Accuracy': [nb_acc, svm_acc, mlp_acc],
    'Test Accuracy (%)': [f"{nb_acc * 100:.2f}%", f"{svm_acc * 100:.2f}%", f"{mlp_acc * 100:.2f}%"],
    'Canceled F1-Score': [0.50, 0.80, 0.84]
}).sort_values(by='Test Accuracy', ascending=False).reset_index(drop=True)

print("=== Model Performance Comparison ===")
print(comparison_df.to_string(index=False))

# Automatically print which model performed best
best_row = comparison_df.iloc[0]
print(f"\nBest Performing Model: {best_row['Model']} with {best_row['Test Accuracy (%)']} test accuracy.")


=== Model Performance Comparison ===
                       Model  Test Accuracy Test Accuracy (%)  Canceled F1-Score
        Neural Network (MLP)       0.917434            91.74%               0.84
Support Vector Machine (SVM)       0.901843            90.18%               0.80
                 Naïve Bayes       0.496811            49.68%               0.50

Best Performing Model: Neural Network (MLP) with 91.74% test accuracy.


### Reflection:
1. Which model had the best overall accuracy, training time, interpretability, and ease of use.
2. Would you recommend this model for deployment, and why?

### ✍️ Your Response: 🔧
1. The Neural Network model had the best overall accuracy coming in at 91.7% and a 0.84 F score. Naive Bayes had the best ease of use and lowest training time. Linear SVM offered the best interpretability because linear feature wights make it very easy to understand which factors most influence cancellations.

2. I would recommend deploying the Neural Network model because it has the best overall accuracy and provides the most financial value. I would try to restrict this model to only back of house functions and if needed for customer-facing decisions, try and pair it with a system that can help more easily interpret the model reasoning.

## 6. Final Business Recommendation

### Reflection:
1. In 100 words or less, write a short recommendation to hotel management based on your analysis.

Possible info to include:
- Which model do you recommend implementing?
- What business problem does it help solve?
- Are there any risks or limitations?
- What additional data might improve the results in the future?

2. How does this relate to your customized learning outcome you created in canvas?


### ✍️ Your Response: 🔧
1. I recommend deploying the Neural Network model, which achieved 91.7% accuracy and caught 85% of cancellations with minimal false alarms. Implementing this model will improve room inventory forecasting, dynamic booking, and housekeeping staffing across the properties. Key risks include its black box nature, making individual predicition hard to explain, and the potential for overbooking of loyal guests if false positives occur. To improve further accuracy, we should incorporate a guest loyalty system which can track individual customer history to further track and train our model on customer behaviors.

2. This assignment supports my goal of building technical skills for my auditing career because evaluating data integrity, internal controls, and model reliability is essential. Detecting and preventing data leakage before splitting datasets mirrors how an auditor needs to ensure completeness, accuracy, and objectivity in financial and operational testing.

## Submission Instructions
✅ Checklist:
- All code cells run without error
- All markdown responses are complete
- Submit on Canvas as instructed

In [ ]:
!jupyter nbconvert --to html "assignment_12_bayes_svm_neural.ipynb"